# `preion.forecast` tutorial

End-to-end walk-through of the forecast pipeline: build a config, generate
mock kSZ/tau/BB data with `make_datapoints`, run a (short, for this notebook)
`emcee` MCMC with `run_mcmc`, then read back and diagnose the
chain the way `preion-read-mcmc` does.

In [1]:
import shutil
import tempfile
from pathlib import Path

from preion.forecast import config, datapoints, mcmc, read_mcmc

%matplotlib inline

could not load wigners.so fortran shared object
try f2py -c -m wigners wigners.f90 from the command line in wigners directory ?


## 1. Load a config

`config.load_config` reads a YAML file and fills in defaults matching the
values that used to be hardcoded in `run_mcmc_cv_limited_new.py`. We start
from the packaged example config and shrink it to a tiny, fast run in a
scratch directory.

In [ ]:
run_dir = Path(tempfile.mkdtemp(prefix='preion_forecast_tutorial_'))

cfg = config.load_config('../configs/cv_limited_new.yaml')
cfg['data'] = 'tau'
cfg['niterations'] = 10
# emcee requires enough walkers relative to ndim (4 here) to keep the initial
# ensemble's covariance well-conditioned.
cfg['nwalkers'] = 8
cfg['output_dir'] = str(run_dir)
cfg

{'label': 'cv_limited_new',
 'data': 'tau',
 'theta_true': [7.0, 1.5, 3.7, 0.1],
 'log_kappa': True,
 'niterations': 10,
 'nwalkers': 8,
 'fsky': 1.0,
 'use_ksz_emulator': 'RF',
 'overwrite': True,
 'plot': False,
 'telescopes': None,
 'output_dir': '/tmp/preion_forecast_tutorial_tg0dm651',
 'ells': {'tau': {'start': 100, 'stop': 5000, 'step': 100},
  'ksz': {'start': 1000, 'stop': 8000, 'step': 500},
  'bb': {'start': 10, 'stop': 1000, 'num': 50}}}

## 2. Generate mock data directly

`mcmc.get_or_make_datapoints` will do this automatically (reading cached
datapoints back from disk if they already exist), but it's useful to see
`make_datapoints` on its own. If `telescopes=None` then the forecast is
cosmic-variance limited.

In [ ]:
ells = config.build_ells(cfg)
tau_ps, ksz_ps, bb_ps, cov_tau, cov_ksz, cov_bb = datapoints.make_datapoints(
    cfg['theta_true'], telescopes=cfg['telescopes'], ells=ells,
    use_ksz_emulator=cfg['use_ksz_emulator'],
)
print('tau_ps:', tau_ps)

## 3. Run the MCMC

`mcmc.get_or_make_datapoints(cfg)` reuses (or regenerates) the mock data and
covariances, and `mcmc.run_mcmc(datapoints, cfg)` runs `emcee` against them
and writes the chain to `{output_dir}/backends/`.

In [ ]:
datapoints_dict = mcmc.get_or_make_datapoints(cfg)
sampler = mcmc.run_mcmc(datapoints_dict, cfg)

## 4. Read back and diagnose the chain

This mirrors what `preion-read-mcmc configs/cv_limited_new.yaml` does on the
command line: load the chain, compute convergence diagnostics, flatten the
post-burn-in samples, and plot a corner/trace/posterior-predictive figure.

In [ ]:
ells, data, cov = read_mcmc.load_mock_data(cfg)
sampler = read_mcmc.load_chain(cfg)
diagnostics = read_mcmc.convergence_diagnostics(sampler)
print(f"Auto-correlation time: {diagnostics['endtau']:.2f}. Converged: {diagnostics['converged']}.")
print(f"burnin = {diagnostics['burnin']} steps")

In [ ]:
flatsamples, logps = read_mcmc.get_flat_samples(sampler, diagnostics['burnin'])
truths = read_mcmc._theta_true(cfg)
labels = read_mcmc._theta_labels(cfg)
summary = read_mcmc.summarize(flatsamples, truths, read_mcmc.PARAM_NAMES)

In [ ]:
fig_corner = read_mcmc.plot_corner(flatsamples, truths, labels)

In [ ]:
fig_trace = read_mcmc.plot_trace(sampler, read_mcmc.PARAM_NAMES, diagnostics['burnin'])

In [ ]:
fig_pp = read_mcmc.plot_posterior_predictive(sampler, ells, data, cov, diagnostics['burnin'])

## 5. Clean up the scratch run directory

In [ ]:
shutil.rmtree(run_dir, ignore_errors=True)